# Customer Churn Prediction - Preprocesamiento y Feature Engineering

**Objetivo:**
Preparar los datos para entrenamiento de modelos de Machine Learning. Esto incluye:

1. Limpieza de datos: manejo de valores faltantes, corrección de tipos de datos.
2. Codificación de variables categóricas para ML.
3. Creación de nuevas features que puedan mejorar la predicción de churn.
4. Separación en datasets de entrenamiento y prueba para modelado futuro.


In [6]:
# Importación de librerías esenciales
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

# Configuración general
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

In [7]:
# Cargar dataset desde la carpeta raw
data = pd.read_csv('../data/raw/Telco_Customer_Churn.csv')

# Copia de trabajo
df = data.copy()

# Mostrar dimensiones y primeras filas
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [8]:
# Comprobamos tipos de datos
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

## 🔹 Limpieza y codificación

En esta sección convertimos `TotalCharges` a numérico, eliminamos los registros con valores faltantes
y codificamos las variables categóricas.
Las variables binarias se convierten a 0/1 y las multicategoría mediante One-Hot Encoding.

Posteriormente, preparamos la variable objetivo `Churn` como binaria.

In [9]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float)
df = df.dropna(subset=['TotalCharges'])

print(f"Dataset shape after cleaning: {df.shape}")

Dataset shape after cleaning: (7032, 21)


In [10]:
# Columnas binarias
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0})

# Columnas con múltiples categorías
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('customerID')
cat_cols.remove('Churn')
multi_cat_cols = [col for col in cat_cols if col not in binary_cols]

# One-Hot Encoding
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

In [11]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

## 🔹 División y escalado

Separamos el dataset en conjuntos de entrenamiento y prueba (80/20) estratificados por `Churn`
y escalamos las variables numéricas (`tenure`, `MonthlyCharges`, `TotalCharges`) usando `StandardScaler`.

In [12]:
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])